<a href="https://colab.research.google.com/github/yujoilly23/CatchmeIfyoucan/blob/main/Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [47]:
base_path = "/content/drive/MyDrive/CATCHMEIFYOUCAN"

video_path = "/content/drive/MyDrive/CATCHMEIFYOUCAN/data/raw.mp4"   # ← 파일명 맞게 수정
output_path = base_path + "/output/output_landmark.mp4"

In [48]:
!pip install dlib imutils

In [33]:
!wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bzip2 -d shape_predictor_68_face_landmarks.dat.bz2

--2026-04-01 08:53:36--  http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Resolving dlib.net (dlib.net)... 107.180.26.78
Connecting to dlib.net (dlib.net)|107.180.26.78|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2 [following]
--2026-04-01 08:53:36--  https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Connecting to dlib.net (dlib.net)|107.180.26.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64040097 (61M)
Saving to: ‘shape_predictor_68_face_landmarks.dat.bz2’

shape_predictor_68_ 100%[===================>]  61.07M  41.0MB/s    in 1.5s    

2026-04-01 08:53:38 (41.0 MB/s) - ‘shape_predictor_68_face_landmarks.dat.bz2’ saved [64040097/64040097]



In [49]:
import os

print(os.path.exists(video_path))

True


In [50]:
import cv2
import dlib

detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

In [51]:
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(3))
height = int(cap.get(4))

print("FPS:", fps)
print("Size:", width, height)

cap.release()

FPS: 24.0
Size: 406 720


In [52]:
cap = cv2.VideoCapture(video_path)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

frame_count = 0
max_frames = int(fps * 3)  # 👉 3초

while cap.isOpened():
    ret, frame = cap.read()
    if not ret or frame_count > max_frames:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = detector(gray)

    for face in faces:
        x1 = face.left()
        y1 = face.top()
        x2 = face.right()
        y2 = face.bottom()

        # 얼굴 박스
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)

        # 랜드마크
        landmarks = predictor(gray, face)

        for i in range(68):
            x = landmarks.part(i).x
            y = landmarks.part(i).y
            cv2.circle(frame, (x, y), 2, (0,0,255), -1)

    out.write(frame)
    frame_count += 1

cap.release()
out.release()

print("완료!")

완료!


In [53]:
from IPython.display import Video

Video(output_path)